# SQL 결과 추론 문제은행

표를 먼저 읽고 JOIN·NULL·집계·상관 서브쿼리를 실제 행 단위로 추적한다.

답안 칸에 먼저 답을 쓰고, 바로 아래의 정답·해설을 펼쳐 확인하세요. #는 오답 표시로 사용하세요.

## 1회 유형 · 심화

기존 문제의 답안 기록은 백업에 보존하고, 이 새 문제은행의 답안 칸은 비웠습니다.

### 이 은행에서 공통으로 쓰는 테이블

<b>학과</b>

| 학과코드 | 학과명 | 단과대 |
|---|---|---|
| C01 | 컴퓨터공학 | 공과대 |
| E01 | 전자공학 | 공과대 |
| B01 | 경영학 | 상경대 |
| M01 | 수학 | 자연대 |

<b>학생</b>

| 학번 | 이름 | 학과코드 | 성적 |
|---|---|---|---|
| 1001 | 김한슬 | C01 | 95 |
| 1002 | 이도현 | C01 | 82 |
| 1003 | 박서준 | C01 | NULL |
| 1004 | 최유진 | E01 | 78 |
| 1005 | 정민재 | E01 | 91 |
| 1006 | 한지우 | B01 | 65 |
| 1007 | 오세영 | B01 | 88 |
| 1008 | 강태윤 | B01 | NULL |
| 1009 | 윤나래 | NULL | 72 |

> 학과 M01(수학)에는 학생이 한 명도 없고, 학생 1009는 학과코드가 NULL이다.
> **NULL과 매칭 없는 행이 답을 가르는 문제가 많다.**


---

> 아래 문제의 결과는 모두 위 표의 데이터로 **실제 실행해 확인한 값**입니다.

---

## Q1. 외래키 제약 빈칸 채우기 (26년1회 1번 유형)

다음은 수강 테이블을 생성하는 SQL이다. 학생 테이블의 학번을 참조하는 외래키를
`FK_수강_학생` 이라는 이름으로 걸려고 한다. 빈칸 ① ~ ⑤를 채우시오.

```sql
CREATE TABLE 수강 (
    수강번호 CHAR(5) NOT NULL,
    학번     CHAR(4) NOT NULL,
    과목명   VARCHAR(30),
    ( ① ) FK_수강_학생 ( ② ) KEY (( ③ )) ( ④ ) 학생(( ⑤ ))
        ON DELETE CASCADE
);
```

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
① CONSTRAINT   ② FOREIGN   ③ 학번   ④ REFERENCES   ⑤ 학번
```

제약조건에 **이름을 붙일 때**는 `CONSTRAINT 이름` 을 앞에 쓴다.

```sql
CONSTRAINT FK_수강_학생 FOREIGN KEY (학번) REFERENCES 학생(학번) ON DELETE CASCADE
```

- ③은 **이 테이블(수강)의 컬럼**, ⑤는 **참조당하는 테이블(학생)의 컬럼**이다. 방향을 바꿔 쓰면 오답이다.
- `ON DELETE CASCADE` : 부모 행이 지워지면 자식 행도 함께 삭제
- `ON DELETE SET NULL` : 자식의 외래키를 NULL로
- `ON DELETE RESTRICT` : 참조 중이면 삭제를 막음

</details>

## Q2. 제약조건 빈칸 채우기

다음 조건을 만족하도록 빈칸 ① ~ ④를 채우시오.

- 사번은 기본키
- 이름은 NULL을 허용하지 않음
- 부서는 값을 안 주면 '미배정'
- 급여는 1000 이상 9000 이하만 허용

```sql
CREATE TABLE 사원 (
    사번 CHAR(5) ( ① ),
    이름 VARCHAR(20) ( ② ),
    부서 VARCHAR(20) ( ③ ) '미배정',
    급여 NUMBER(5) ( ④ ) (급여 >= 1000 AND 급여 <= 9000)
);
```

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
① PRIMARY KEY   ② NOT NULL   ③ DEFAULT   ④ CHECK
```

제약조건 6가지는 **PRIMARY KEY / FOREIGN KEY / UNIQUE / NOT NULL / CHECK / DEFAULT** 다.

- `UNIQUE` 는 중복만 막고 **NULL은 허용**한다. `PRIMARY KEY` 는 중복도 NULL도 막는다. 이 차이가 자주 나온다.
- `CHECK (급여 BETWEEN 1000 AND 9000)` 으로 써도 같은 뜻이다.

</details>

## Q3. DISTINCT 와 결과 튜플 수 (26년1회 13번 유형)

다음 세 SQL의 결과 튜플(행) 수를 각각 쓰시오.

```sql
① SELECT 학과코드 FROM 학생;
② SELECT DISTINCT 학과코드 FROM 학생;
③ SELECT COUNT(DISTINCT 학과코드) FROM 학생 WHERE 학과코드 = 'C01';
```

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
① | ② | ③
--+---+--
9 | 4 | 1
```

- ① 전체 행 수 그대로 → **9**
- ② 중복 제거 : C01, E01, B01, 그리고 **NULL도 하나의 값으로 세어** → **4**
- ③ `COUNT(...)` 같은 집계함수는 그룹이 없으면 **결과가 언제나 한 줄**이다 → **1**

③에서 "1"이라는 **값**과 "1개"라는 **튜플 수**를 헷갈리면 안 된다. 문제는 튜플 수를 물었다.

</details>

## Q4. COUNT · AVG 와 NULL 처리

다음 SQL의 실행 결과를 쓰시오.

```sql
SELECT COUNT(*), COUNT(성적), SUM(성적), AVG(성적)
FROM 학생;
```

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
COUNT(*) | COUNT(성적) | SUM(성적) | ROUND(AVG(성적),2)
---------+-----------+---------+-----------------
9        | 7         | 571     | 81.57           
```

- `COUNT(*)` : NULL 포함 **전체 행** → 9
- `COUNT(성적)` : **NULL을 제외**한 개수 → 7
- `SUM`, `AVG` 도 **NULL을 계산에서 제외**한다. 따라서 평균의 분모는 9가 아니라 **7**이다.

571 / 7 = 81.57… 이다. NULL을 0으로 취급해 9로 나누면 63.4가 되는데 **오답**이다.

</details>

## Q5. GROUP BY 와 HAVING

다음 SQL의 실행 결과를 쓰시오.

```sql
SELECT 학과코드, COUNT(*) AS 인원, AVG(성적) AS 평균
FROM 학생
WHERE 성적 IS NOT NULL
GROUP BY 학과코드
HAVING COUNT(*) >= 2
ORDER BY 평균 DESC;
```

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
학과코드 | 인원 | 평균  
-----+----+-----
C01  | 2  | 88.5
E01  | 2  | 84.5
B01  | 2  | 76.5
```

실행 순서를 따라가야 한다 : **FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY**

1. `WHERE 성적 IS NOT NULL` 로 박서준·강태윤이 먼저 빠진다 (**그룹화 전**)
2. 학과코드로 묶으면 C01 2명, E01 2명, B01 2명, NULL 1명
3. `HAVING COUNT(*) >= 2` 로 학과코드가 NULL인 그룹(1명)이 탈락
4. 평균 내림차순 정렬

**WHERE는 행 단위, HAVING은 그룹 단위.** 집계함수 조건은 HAVING에만 쓸 수 있다.

</details>

## Q6. JOIN 과 서브쿼리 (26년1회 8번 유형)

다음 SQL의 실행 결과를 쓰시오.

```sql
SELECT COUNT(*)
FROM 학생 s
JOIN 학과 d ON s.학과코드 = d.학과코드
WHERE d.단과대 = '공과대'
  AND s.성적 > (SELECT AVG(성적) FROM 학생);
```

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
COUNT(*)
--------
3       
```

1. 서브쿼리 `AVG(성적)` = 571/7 = **81.57…** (NULL 제외)
2. 공과대(C01, E01) 학생 중 성적이 81.57보다 큰 사람 : 김한슬(95), 이도현(82), 정민재(91)
3. 최유진(78)은 미달, 박서준은 성적이 NULL이라 **비교 자체가 참이 될 수 없다**

→ **3**

NULL은 어떤 값과 비교해도 참이 아니라는 점이 함정이다.

</details>

## Q7. LEFT OUTER JOIN 과 매칭 없는 행

다음 SQL의 실행 결과를 쓰시오.

```sql
SELECT d.학과명, COUNT(s.학번) AS 인원
FROM 학과 d
LEFT OUTER JOIN 학생 s ON d.학과코드 = s.학과코드
GROUP BY d.학과명
ORDER BY 인원 DESC, d.학과명;
```

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
학과명   | 인원
------+---
경영학   | 3 
컴퓨터공학 | 3 
전자공학  | 2 
수학    | 0 
```

`LEFT OUTER JOIN` 은 **왼쪽(학과) 행을 전부 남긴다.** 학생이 없는 수학과도 결과에 나온다.

핵심은 수학과의 인원이 **1이 아니라 0** 이라는 것이다.
`COUNT(s.학번)` 은 매칭 실패로 생긴 **NULL을 세지 않기** 때문이다.
만약 `COUNT(*)` 였다면 행 자체는 1개라서 **1**이 나온다. 이 차이가 자주 출제된다.

또 학생 1009(윤나래)는 학과코드가 NULL이라 **어느 학과에도 붙지 않아** 결과에서 빠진다.

</details>

## Q8. EXISTS 상관 서브쿼리

다음 SQL의 실행 결과를 쓰시오.

```sql
SELECT 학과명
FROM 학과 d
WHERE NOT EXISTS (
    SELECT 1 FROM 학생 s WHERE s.학과코드 = d.학과코드
);
```

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
학과명
---
수학 
```

`EXISTS` 는 **행이 하나라도 있으면 참**이다. 값이 무엇인지는 보지 않기 때문에 `SELECT 1` 로 쓴다.
`NOT EXISTS` 는 그 반대이므로 **학생이 한 명도 없는 학과**를 찾는다 → 수학.

`IN` 은 값 목록과 비교하지만 `EXISTS` 는 **존재 여부만** 본다.
서브쿼리에 NULL이 섞이면 `NOT IN` 은 결과가 통째로 비는 반면 `NOT EXISTS` 는 정상 동작한다.

</details>

## Q9. ALL 과 ANY

다음 두 SQL의 결과를 각각 쓰시오.

```sql
① SELECT COUNT(*) FROM 학생
   WHERE 성적 > ALL (SELECT 성적 FROM 학생 WHERE 학과코드 = 'B01' AND 성적 IS NOT NULL);

② SELECT COUNT(*) FROM 학생
   WHERE 성적 > ANY (SELECT 성적 FROM 학생 WHERE 학과코드 = 'B01' AND 성적 IS NOT NULL);
```

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
① | ②
--+--
2 | 6
```

서브쿼리 결과는 경영학과 성적 **{65, 88}** 이다.

- `> ALL` : **가장 큰 값(88)보다 커야** 한다 → 95, 91 → **2명**
- `> ANY` : **가장 작은 값(65)보다 크기만** 하면 된다 → 95, 82, 78, 91, 88, 72 → **6명**

`> ALL` = 최댓값보다 크다, `> ANY` = 최솟값보다 크다. 이 치환을 외워두면 빠르다.

</details>

## Q10. UNION 과 UNION ALL

다음 두 SQL의 결과 행 수를 각각 쓰시오.

```sql
① SELECT 학과코드 FROM 학생 WHERE 성적 >= 80
   UNION
   SELECT 학과코드 FROM 학생 WHERE 성적 < 80;

② SELECT 학과코드 FROM 학생 WHERE 성적 >= 80
   UNION ALL
   SELECT 학과코드 FROM 학생 WHERE 성적 < 80;
```

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
① | ②
--+--
4 | 7
```

성적 80 이상은 김한슬·이도현·정민재·오세영(C01,C01,E01,B01),
80 미만은 최유진·한지우·윤나래(E01,B01,NULL). 성적이 NULL인 두 명은 **양쪽 모두에서 빠진다.**

- `UNION` : **중복을 제거**한다 → C01, E01, B01, NULL → **4**
- `UNION ALL` : 중복을 그대로 둔다 → 4 + 3 = **7**

`UNION` 은 내부적으로 정렬·중복제거를 하므로 느리다. 중복이 없다고 확신하면 `UNION ALL` 을 쓴다.

</details>

## Q11. NULL 처리 함수

다음 SQL의 실행 결과를 쓰시오.

```sql
SELECT 이름, COALESCE(성적, 0) AS 점수
FROM 학생
WHERE 학과코드 = 'C01'
ORDER BY 점수;
```

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
이름  | 점수
----+---
박서준 | 0 
이도현 | 82
김한슬 | 95
```

`COALESCE(값, 대체값)` 은 표준 SQL, Oracle의 `NVL(값, 대체값)` 과 같은 역할이다.
NULL이던 박서준의 성적이 0으로 바뀌어 정렬에서 맨 앞에 온다.

**주의**: `WHERE 성적 = NULL` 은 절대 참이 되지 않는다. 반드시 `IS NULL` / `IS NOT NULL` 을 쓴다.

</details>

## Q12. 상위 N건 구하기

다음 SQL의 실행 결과를 쓰시오.

```sql
SELECT 이름, 성적
FROM 학생
WHERE 성적 IS NOT NULL
ORDER BY 성적 DESC
LIMIT 3;
```

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
이름  | 성적
----+---
김한슬 | 95
정민재 | 91
오세영 | 88
```

상위 N건을 뽑는 문법은 DBMS마다 다르다. **어느 쪽이 나와도 읽을 수 있어야 한다.**

| DBMS | 문법 |
|---|---|
| 표준 / MySQL / SQLite | `LIMIT 3` |
| Oracle | `WHERE ROWNUM <= 3` 또는 `FETCH FIRST 3 ROWS ONLY` |
| SQL Server | `SELECT TOP 3 ...` |

Oracle의 `ROWNUM` 은 **정렬 전에 매겨지므로** 서브쿼리로 먼저 정렬해야 한다는 게 함정이다.

</details>

## Q13. 같은 학과 학생 짝짓기 (SELF JOIN)

다음 SQL의 실행 결과를 쓰시오.

```sql
SELECT a.이름, b.이름
FROM 학생 a JOIN 학생 b
  ON a.학과코드 = b.학과코드 AND a.학번 < b.학번
WHERE a.학과코드 = 'C01';
```

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
이름1 | 이름2
----+----
김한슬 | 이도현
김한슬 | 박서준
이도현 | 박서준
```

같은 테이블을 두 번 부르는 **SELF JOIN**이다. 별칭(a, b)이 반드시 필요하다.

`a.학번 < b.학번` 조건이 없으면 자기 자신과의 짝(김한슬-김한슬)과
순서만 바뀐 중복(김한슬-이도현, 이도현-김한슬)까지 전부 나온다.
C01 3명 중 2명을 뽑는 조합이므로 **3쌍**이다.

</details>

## Q14. VIEW 생성과 특징

성적이 85점 이상인 학생의 학번·이름·성적으로 구성된 `우수학생` 뷰를 만드는 SQL을 쓰시오.
또한 뷰에 대한 다음 설명 중 옳은 것을 모두 고르시오.

```
(가) 뷰는 실제 데이터를 저장하는 물리적 테이블이다.
(나) 뷰의 정의를 변경하려면 ALTER VIEW 를 사용한다.
(다) 뷰를 삭제해도 기반 테이블은 영향을 받지 않는다.
(라) 뷰를 통해 데이터를 조회하면 논리적 독립성이 높아진다.
```

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
CREATE VIEW 우수학생 AS
SELECT 학번, 이름, 성적
FROM 학생
WHERE 성적 >= 85;

옳은 것 : (다), (라)
```

- (가) 틀림 — 뷰는 **가상 테이블**이다. 정의(SELECT문)만 저장하고 데이터는 갖지 않는다.
- (나) 틀림 — **ALTER VIEW는 없다.** 정의를 바꾸려면 `DROP VIEW` 후 다시 `CREATE VIEW` 해야 한다.
   (`CREATE OR REPLACE VIEW` 를 지원하는 DBMS도 있다.)
- (다) 맞음 — 뷰를 지워도 기반 테이블은 그대로다. 반대로 **기반 테이블을 지우면 뷰는 못 쓴다.**
- (라) 맞음 — 뷰의 대표적인 장점이 **논리적 독립성**과 **보안**(필요한 열만 노출)이다.

</details>

## Q15. GRANT 와 REVOKE

다음을 수행하는 SQL을 각각 쓰시오.

- ① 사용자 KIM에게 학생 테이블의 조회·삽입 권한을 부여하되, KIM이 그 권한을 다른 사람에게 다시 줄 수 있게 하시오.
- ② 사용자 KIM에게서 학생 테이블의 삽입 권한을 회수하되, KIM이 다른 사람에게 준 권한까지 함께 회수하시오.

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
① GRANT SELECT, INSERT ON 학생 TO KIM WITH GRANT OPTION;

② REVOKE INSERT ON 학생 FROM KIM CASCADE;
```

**GRANT 는 TO, REVOKE 는 FROM.** 전치사를 바꿔 쓰는 실수가 가장 많다.

| 옵션 | 의미 |
|---|---|
| `WITH GRANT OPTION` | 받은 권한을 **남에게 다시 줄 수 있음** |
| `CASCADE` | 회수할 때 **연쇄적으로 함께 회수** |
| `RESTRICT` | 남에게 준 권한이 있으면 **회수를 거부** |

세미콜론까지 정확히 쓴다. 실기는 손으로 쓰는 시험이라 문장부호도 채점 대상이다.

</details>

## Q16. DELETE · TRUNCATE · DROP 비교

다음 표의 빈칸 ① ~ ⑥을 채우시오.

| 구분 | DELETE | TRUNCATE | DROP |
|---|---|---|---|
| SQL 분류 | ( ① ) | ( ② ) | DDL |
| 삭제 대상 | ( ③ ) | 전체 행 | ( ④ ) |
| ROLLBACK | ( ⑤ ) | 불가 | 불가 |
| WHERE 절 | 사용 가능 | ( ⑥ ) | 사용 불가 |

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
① DML   ② DDL   ③ 조건에 맞는 행(일부 행)   ④ 테이블 구조 자체   ⑤ 가능   ⑥ 사용 불가
```

세 명령의 차이는 거의 매 회차 나온다.

- **DELETE** 는 유일하게 **DML** 이라 트랜잭션에 묶이고 `ROLLBACK` 이 된다. 느리다.
- **TRUNCATE** 는 DDL이라 **자동 커밋**되어 되돌릴 수 없다. 구조는 남기고 데이터만 전부 지운다. 빠르다.
- **DROP** 은 테이블 자체를 없앤다.

`DROP TABLE 학생 CASCADE` 는 참조하는 것까지 함께 삭제, `RESTRICT` 는 참조 중이면 삭제를 거부한다.

</details>

## Q17. 인덱스 생성과 특성

① 학생 테이블의 이름 컬럼에 `idx_학생_이름` 이라는 인덱스를 만드는 SQL을 쓰시오.
② 값의 중복을 허용하지 않는 인덱스를 만들려면 어떤 키워드를 추가해야 하는가?
③ 인덱스를 많이 만들면 오히려 성능이 나빠지는 작업은 무엇인가? (3가지)

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
① CREATE INDEX idx_학생_이름 ON 학생(이름);

② UNIQUE  →  CREATE UNIQUE INDEX ...

③ INSERT, UPDATE, DELETE
```

인덱스는 **검색(SELECT)은 빠르게** 하지만, 데이터가 바뀔 때마다 **인덱스도 같이 갱신**해야 해서
`INSERT` / `UPDATE` / `DELETE` 는 오히려 느려진다. 저장 공간도 추가로 쓴다.

- 카디널리티가 **높은**(중복이 적은) 컬럼일수록 인덱스 효과가 크다.
- 삭제는 `DROP INDEX idx_학생_이름;`

</details>

## Q18. 트리거 구성 요소

다음은 학생 테이블에 행이 추가된 뒤 로그를 남기는 트리거다. 빈칸 ① ~ ④를 채우시오.

```sql
CREATE ( ① ) 학생_입력로그
( ② ) INSERT ON 학생
FOR EACH ( ③ )
BEGIN
    INSERT INTO 로그 VALUES (:( ④ ).학번, SYSDATE);
END;
```

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
① TRIGGER   ② AFTER   ③ ROW   ④ NEW
```

- ② **AFTER** : 작업이 끝난 **뒤** 실행. 검증 목적이면 `BEFORE` 를 쓴다.
- ③ `FOR EACH ROW` : **행 단위 트리거**. 생략하면 문장 단위(statement) 트리거가 된다.
- ④ `:NEW` 는 **새로 들어온 값**, `:OLD` 는 **변경 전 값**이다.
  INSERT에는 `:NEW` 만, DELETE에는 `:OLD` 만 있고, UPDATE에는 둘 다 있다.

**트리거 안에서는 COMMIT / ROLLBACK 을 쓸 수 없다.** 자주 출제되는 함정이다.

</details>

## Q19. 프로시저와 함수의 차이

저장 프로시저(Stored Procedure)와 사용자 정의 함수(Function)의 차이를
① 반환값, ② 호출 방법, ③ SELECT 문 안에서의 사용 가능 여부 세 가지로 구분해 쓰시오.
또한 프로시저 매개변수의 3가지 종류를 쓰시오.

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
① 반환값   : 함수는 RETURN으로 반드시 값을 하나 반환한다. 프로시저는 반환값이 없어도 된다.
② 호출     : 프로시저는 CALL(또는 EXECUTE)로 호출, 함수는 식(expression) 안에서 호출한다.
③ SELECT 내 사용 : 함수는 가능, 프로시저는 불가능.

매개변수 3종 : IN(입력), OUT(출력), INOUT(입출력)
```

한 줄 요약 — **함수는 값을 만들어 돌려주는 것, 프로시저는 일을 시키는 것.**

그래서 함수는 `SELECT 함수명(컬럼) FROM ...` 처럼 식 안에 넣을 수 있지만
프로시저는 `CALL 프로시저명(인자);` 로 따로 실행해야 한다.

</details>

## Q20. 자주 틀리는 지점 종합

다음 SQL에는 오류가 하나씩 있다. 각각 무엇이 잘못되었는지 지적하고 고쳐 쓰시오.

```sql
① SELECT * FROM 학생 WHERE 성적 = NULL;
② SELECT 학과코드, COUNT(*) FROM 학생 WHERE COUNT(*) >= 2 GROUP BY 학과코드;
③ ALTER TABLE 학생 DROP 성적;
④ SELECT 이름 FROM 학생 WHERE 이름 LIKE "김%";
```

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
① NULL은 = 로 비교할 수 없다.  →  WHERE 성적 IS NULL

② 집계함수는 WHERE에 쓸 수 없다.  →  GROUP BY 학과코드 HAVING COUNT(*) >= 2

③ 컬럼 삭제에는 COLUMN 키워드가 필요하다.  →  ALTER TABLE 학생 DROP COLUMN 성적;

④ 문자열은 작은따옴표를 쓴다.  →  WHERE 이름 LIKE '김%'
```

이 네 가지가 실기에서 가장 많이 나오는 감점 포인트다.

- NULL 비교는 **언제나 `IS NULL` / `IS NOT NULL`**
- **WHERE는 그룹화 전**이라 집계함수를 모른다. 집계 조건은 반드시 **HAVING**
- `ALTER TABLE ... ADD 컬럼명 타입` / `MODIFY` / `DROP COLUMN` — DROP에만 COLUMN이 붙는다
- SQL의 문자열은 **작은따옴표**. 큰따옴표는 식별자(컬럼·테이블 이름)용이다

추가로 자주 틀리는 것 : `BETWEEN` 은 **양 끝값 포함**, `ORDER BY` 기본은 **ASC**, 세미콜론 누락.

</details>

## 2회 유형 · 기본/중간

기존 문제의 답안 기록은 백업에 보존하고, 이 새 문제은행의 답안 칸은 비웠습니다.

## Q1. 상관 서브쿼리 (26년2회 8번 유형)

**[A] 테이블**

| id | x |
|---|---|
| 1 | 10 |
| 2 | 20 |
| 3 | 30 |
| 4 | 40 |

**[B] 테이블**

| id | y |
|---|---|
| 1 | 5 |
| 1 | 15 |
| 2 | 20 |
| 3 | 35 |
| 5 | 50 |

```sql
SELECT COUNT(*)
FROM A
WHERE x > (
    SELECT AVG(y)
    FROM B
    WHERE B.id IN (
        SELECT A2.id
        FROM A A2
        WHERE A2.x < A.x
    )
);
```

**실행 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
COUNT(*)
--------
3       
```

**상관 서브쿼리**다 — 안쪽 쿼리가 바깥의 `A.x` 를 참조하므로 **A의 행마다 따로 계산**해야 한다.
한 번에 풀려 하지 말고 4행을 각각 표로 정리한다.

| A.x | 나보다 작은 A2.id | B에서 뽑히는 y | AVG(y) | x > AVG? |
|---|---|---|---|---|
| 10 | 없음 | 없음 | **NULL** | ❌ |
| 20 | 1 | 5, 15 | 10 | ✅ |
| 30 | 1, 2 | 5, 15, 20 | 13.33 | ✅ |
| 40 | 1, 2, 3 | 5, 15, 20, 35 | 18.75 | ✅ |

→ **3**

첫 행이 핵심이다. 서브쿼리가 **빈 집합**이면 `AVG` 는 0이 아니라 **NULL** 이고,
`10 > NULL` 은 참이 아니라 **알 수 없음(UNKNOWN)** 이라 WHERE를 통과하지 못한다.

</details>

## Q2. OUTER JOIN + IS NULL — 안티 조인 (26년2회 15번 유형)

**[A] 테이블**

| id | v |
|---|---|
| 1 | 10 |
| 1 | 20 |
| 2 | 30 |

**[B] 테이블**

| id | w |
|---|---|
| 1 | 100 |
| 3 | 300 |
| 4 | 400 |

```sql
SELECT COUNT(*) AS Result
FROM A
RIGHT OUTER JOIN B ON A.id = B.id
WHERE A.id IS NULL;
```

**실행 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
Result
------
1     
```

`RIGHT OUTER JOIN B` 이므로 **B의 모든 행이 남는다.** 조인 결과를 먼저 그린다.

| B.id | 매칭되는 A 행 | 결과 행 수 | A.id |
|---|---|---|---|
| 1 | (1,10), (1,20) 두 개 | 2 | 1 |
| 3 | 없음 | 1 | **NULL** |
| 4 | 없음 | 1 | **NULL** |

`WHERE A.id IS NULL` 은 **매칭에 실패한 행만** 남긴다 → id 3, 4 → **2**

이 패턴(OUTER JOIN + IS NULL)을 **안티 조인**이라 하며,
"B에는 있는데 A에는 없는 것"을 찾는 표준 관용구다. `NOT IN` / `NOT EXISTS` 와 같은 목적이다.

> B.id = 1이 **2행으로 늘어나는 것**도 함정이다. 조인은 매칭되는 만큼 행이 곱해진다.

</details>

## Q3. 도메인 무결성 — CREATE DOMAIN (26년2회 14번 유형)

다음 SQL은 SEASON 도메인의 값이 지정된 계절 중 하나만 되도록 **도메인 무결성**을 보장한다.
빈칸 ①에 들어갈 SQL 키워드를 쓰시오.

```sql
CREATE DOMAIN SEASON AS VARCHAR(6)
    ( ① ) (VALUE IN ('spring', 'summer', 'autumn', 'winter'));
```

또한 무결성 제약의 3가지 종류를 쓰고 각각을 한 줄로 설명하시오.

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
① CHECK

무결성 3종
- 개체 무결성 : 기본키는 NULL이 될 수 없고 중복될 수 없다
- 참조 무결성 : 외래키는 참조하는 릴레이션의 기본키 값이거나 NULL이어야 한다
- 도메인 무결성 : 각 속성의 값은 정의된 도메인(자료형·범위·형식)에 속해야 한다
```

`CREATE DOMAIN` 은 **재사용 가능한 사용자 정의 자료형**을 만드는 구문이고,
값의 범위를 제한하는 키워드는 테이블 제약과 똑같이 **CHECK** 다.

`VALUE` 는 그 도메인에 들어올 값을 가리키는 예약어다 (컬럼명이 아니다).

```sql
CREATE DOMAIN SEASON AS VARCHAR(6)
    CHECK (VALUE IN ('spring','summer','autumn','winter'));
```

> 도메인 제약은 **도메인 무결성**에 해당한다. 셋 중 무엇인지 함께 묻는 경우가 많다.

</details>

## Q4. LIKE 와 ORDER BY 빈칸 (26년2회 19번 유형)

**[회사] 테이블**

| 이름 | 부서 | 연봉 |
|---|---|---|
| 이서연 | 영업부 | 6200 |
| 박도윤 | 영업부 | 4800 |
| 이하준 | 기획부 | 3400 |
| 최지우 | 총무부 | 3100 |
| 이유나 | 기획부 | 5300 |

이름이 '이'로 시작하는 사원을 **이름의 내림차순**으로 조회하려 한다. 빈칸을 채우시오.

```sql
SELECT *
FROM 회사
WHERE 이름 LIKE '( ① )'
ORDER BY 이름 ( ② );
```

그리고 **실행 결과**도 쓰시오.

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
① 이%
② DESC

실행 결과

이름  | 부서  | 연봉  
----+-----+-----
이하준 | 기획부 | 3400
이유나 | 기획부 | 5300
이서연 | 영업부 | 6200
```

- ① **`이%`** — `%` 는 **0글자 이상** 아무 문자열. '이'로 시작하기만 하면 된다.
- ② **`DESC`** — 내림차순. 생략하면 기본값이 `ASC`(오름차순)다.

와일드카드를 구분할 것.

| 패턴 | 뜻 |
|---|---|
| `'이%'` | 이로 **시작** |
| `'%이'` | 이로 **끝남** |
| `'%이%'` | 이를 **포함** |
| `'이_'` | 이 + **정확히 한 글자** (두 글자 이름만) |

한글 내림차순은 가나다 역순이라 이하준 → 이유나 → 이서연 순이 된다.

</details>

## Q5. 집계함수와 NULL — 빈 집합의 AVG

**[B] 테이블** (위 문제와 동일)

| id | y |
|---|---|
| 1 | 5 |
| 1 | 15 |
| 2 | 20 |
| 3 | 35 |
| 5 | 50 |

```sql
SELECT COUNT(*), COUNT(y), AVG(y),
       (SELECT AVG(y) FROM B WHERE id = 99) AS 빈집합
FROM B;
```

**실행 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
COUNT(*) | COUNT(y) | AVG(y) | 빈집합 
---------+----------+--------+-----
5        | 5        | 25.0   | NULL
```

집계함수가 **조회할 행이 하나도 없을 때** 무엇을 반환하는지가 핵심이다.

| 함수 | 빈 집합일 때 |
|---|---|
| `COUNT(*)` / `COUNT(컬럼)` | **0** |
| `SUM` / `AVG` / `MAX` / `MIN` | **NULL** |

`COUNT` 만 0이고 나머지는 전부 **NULL** 이다. 0이 아니다.

그래서 `x > (SELECT AVG(...) ...)` 같은 비교에서 서브쿼리가 비면
`x > NULL` 이 되어 **참이 될 수 없다** — 26년 2회 8번의 첫 행이 정확히 이 경우였다.

</details>